# 🧩 Notebook 3: Real-World Patterns

Circuit breakers rarely live alone. In real systems you combine them with:
- **retries** (for transient blips),
- **fallbacks** (for graceful degradation),
- **metrics** (so you can *see* the breaker working).

This notebook is a short tour of how those fit together.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 1) Retry + breaker — the right order

These two are composed, and the order matters. There are only two arrangements:

**`retry( breaker( call ) )` — breaker on the inside ✅ (the standard choice)**
Every individual attempt passes through the breaker. As soon as the downstream is
truly down the breaker opens, and the *very next* retry fails instantly instead of
burning another timeout. This is what Polly, Resilience4j and Envoy do by default.

> One rule makes it work: **the retry policy must not retry the breaker's own
> "circuit is OPEN" error.** If it does, you spin through all your attempts in a
> microsecond and, worse, keep hammering the probe every time the cool-down expires.
> Treat `CircuitBreakerOpen` as a terminal error and give up.

**`breaker( retry( call ) )` — breaker on the outside ⚠️**
The breaker only ever sees the *final* outcome of a whole retry sequence, so it counts
one failure per logical call instead of one per attempt. With `thr=5` and 4 retries
each, the downstream eats **20** requests before the breaker notices anything is wrong.
Occasionally useful (it hides single blips from the breaker) but it reacts far too
slowly to a real outage.

The code below implements the recommended `retry( breaker( call ) )` arrangement.

In [ ]:
import time, threading, random

class CircuitBreaker:
    def __init__(self, thr=3, reset=1.0):
        self.state='CLOSED'; self.f=0; self.opened=0.0
        self.thr=thr; self.reset=reset; self.lock=threading.Lock()
    def call(self, fn, *a, **kw):
        with self.lock:
            if self.state == 'OPEN':
                if time.time()-self.opened < self.reset:
                    raise RuntimeError('OPEN')
                self.state = 'HALF_OPEN'
        try:
            r = fn(*a, **kw)
            with self.lock: self.state='CLOSED'; self.f=0
            return r
        except Exception:
            with self.lock:
                self.f += 1
                if self.state=='HALF_OPEN' or self.f >= self.thr:
                    self.state='OPEN'; self.opened=time.time()
            raise

def retry_with_backoff(fn, attempts=4, base=0.05):
    for i in range(1, attempts+1):
        try:
            return fn()
        except RuntimeError as e:
            # If the breaker is OPEN, don't waste more retries — bail.
            if 'OPEN' in str(e):
                print(f'  attempt {i}: breaker OPEN → give up')
                raise
            sleep = base * (2 ** (i-1)) * random.uniform(0.5, 1.5)
            print(f'  attempt {i}: {e} → sleep {sleep:.2f}s')
            time.sleep(sleep)
    raise RuntimeError('exhausted retries')

cb = CircuitBreaker(thr=3, reset=1.0)

# downstream fails the first 2 calls, then works
state = {'n': 0}
def sometimes_ok():
    state['n'] += 1
    if state['n'] <= 2:
        raise RuntimeError('transient')
    return 'ok'

print('Case A: transient failures — retry saves the day')
random.seed(0)
print('  result:', retry_with_backoff(lambda: cb.call(sometimes_ok)))

print('\nCase B: downstream is dead — breaker short-circuits retries')
def always_bad():
    raise RuntimeError('down')
cb2 = CircuitBreaker(thr=3, reset=5.0)
try:
    retry_with_backoff(lambda: cb2.call(always_bad), attempts=10)
except Exception as e:
    print('  final:', e)


## 2) Observability — log every state change

You can't tune a breaker you can't see. At minimum, emit a log/metric on every state transition plus a running failure count. In production you'd push these to Prometheus / Datadog / etc.

In [ ]:
class ObservableBreaker(CircuitBreaker):
    def __init__(self, name, **kw):
        super().__init__(**kw)
        self.name = name
        self.metrics = {'opens': 0, 'closes': 0, 'half_opens': 0,
                        'fast_fails': 0, 'calls': 0, 'errors': 0}

    def _transition(self, new):
        if new == self.state: return
        print(f'[breaker:{self.name}] {self.state} -> {new}')
        self.state = new
        if new == 'OPEN': self.metrics['opens'] += 1
        if new == 'HALF_OPEN': self.metrics['half_opens'] += 1
        if new == 'CLOSED': self.metrics['closes'] += 1

    def call(self, fn, *a, **kw):
        self.metrics['calls'] += 1
        with self.lock:
            if self.state == 'OPEN':
                if time.time()-self.opened < self.reset:
                    self.metrics['fast_fails'] += 1
                    raise RuntimeError('OPEN')
                self._transition('HALF_OPEN')
        try:
            r = fn(*a, **kw)
            with self.lock:
                self._transition('CLOSED'); self.f = 0
            return r
        except Exception:
            self.metrics['errors'] += 1
            with self.lock:
                self.f += 1
                if self.state == 'HALF_OPEN' or self.f >= self.thr:
                    self._transition('OPEN'); self.opened = time.time()
            raise

cb = ObservableBreaker('payments', thr=3, reset=0.5)

for i in range(6):
    try: cb.call(lambda: (_ for _ in ()).throw(RuntimeError('boom')))
    except Exception: pass

time.sleep(0.6)
try: cb.call(lambda: 'ok')
except Exception: pass

print('\nmetrics:', cb.metrics)


## 3) Real libraries (production-ready)

Writing your own breaker is great for learning — in production, use a battle-tested library. Popular options:

| Language | Library | Notes |
|---|---|---|
| Python  | [`pybreaker`](https://github.com/danielfm/pybreaker) | simple, thread-safe, event listeners for metrics |
| Python (async) | [`aiobreaker`](https://pypi.org/project/aiobreaker/) | asyncio-friendly fork |
| Java / JVM | [Resilience4j](https://resilience4j.readme.io/) | modern, modular (breaker + retry + bulkhead) |
| .NET | [Polly](https://www.pollydocs.org/) | breakers, retries, timeouts, bulkheads |
| Go | [`gobreaker`](https://github.com/sony/gobreaker) | tiny, by Sony |
| Service mesh | Istio / Envoy | breaker *outside* your code, at the network edge |

A tiny `pybreaker` sketch (no install needed here — this is just for reference):

```python
import pybreaker

breaker = pybreaker.CircuitBreaker(fail_max=5, reset_timeout=30)

@breaker
def charge_card(token, amount):
    return payments_api.charge(token, amount)  # raises on failure

try:
    charge_card(tok, 42)
except pybreaker.CircuitBreakerError:
    queue_for_later(tok, 42)  # fallback
```

## 4) When to use a circuit breaker — and when not to

### ✅ Reach for one when
- You call a **remote dependency** you don't control (another team's service, a
  third-party API, a database behind a network).
- The failure mode is **slow** as well as broken — a timeout is expensive, so paying
  it on every request is the thing you're trying to avoid.
- You have somewhere useful to go when it opens: a **cache, a default, a degraded
  view, a queue for later**. Fast-failing into a 500 is only half a win.
- The dependency is **shared**: one breaker per (service, dependency) pair protects
  every caller in the process at once.

### 🚫 Don't bother — or actively avoid — when
- **The bug is in your own service.** A breaker on a local function call protects
  nothing; it just hides the stack trace.
- **A steady low error rate is normal.** If 1% of calls always fail, a breaker either
  never trips or flaps constantly. Fix the 1%.
- **The call is in-process or free.** No timeout to save, no pool to protect.
- **Every failure must be surfaced exactly.** Fast-failing turns a specific downstream
  error into a generic "circuit open", which can be worse for debugging *and* for
  the caller's own retry logic.
- **Write-heavy flows without idempotency.** Fast-failing mid-retry can leave state
  half-written. Pair with the `04-patterns/idempotency` lab first.
- **You have no timeout in front of it.** A breaker counts errors; a merely slow
  dependency produces none. See Notebook 1 — timeout first, breaker second.

### Related labs
- `05-microservices/retry` — backoff + jitter
- `05-microservices/bulkhead` — isolate resource pools per dependency
- `04-patterns/idempotency` — safely retry writes